# Análise exploratória (EDA) — Evasão e Sucesso Acadêmico

Card A1 · Integrante A. Dataset: *Predict Students' Dropout and Academic Success* (UCI, id 697).

Execute as células na ordem, a partir da raiz do repositório (ou de `notebooks/`). Os gráficos são salvos em `resultados/graficos/` com prefixo `eda_`.
Os comentários explicam cada decisão, pois elas vão para o relatório. Nenhum número deste notebook foi escrito à mão: tudo vem da execução.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Acha a raiz do repositório, rode de onde rodar
RAIZ = Path.cwd()
if not (RAIZ / "data").exists():
    RAIZ = RAIZ.parent
CSV = RAIZ / "data" / "raw" / "data.csv"
GRAFICOS = RAIZ / "resultados" / "graficos"
GRAFICOS.mkdir(parents=True, exist_ok=True)

plt.rcParams["figure.dpi"] = 100
plt.rcParams["savefig.bbox"] = "tight"

## 1. Carga e visão geral

In [ ]:
# O arquivo usa ponto e vírgula; alguns nomes de coluna vêm com espaço/tab sobrando
df = pd.read_csv(CSV, sep=";")
df.columns = [c.strip() for c in df.columns]

print("Amostras x colunas:", df.shape)
print("Atributos (sem o Target):", df.shape[1] - 1)
print("Duplicatas:", df.duplicated().sum())
df.head()

In [ ]:
# Tipos de dados e valores ausentes por coluna
resumo = pd.DataFrame({
    "tipo": df.dtypes.astype(str),
    "ausentes": df.isna().sum(),
    "valores_unicos": df.nunique(),
})
print("Total de valores ausentes:", int(df.isna().sum().sum()))
resumo

## 2. Target e desbalanceamento

In [ ]:
contagem = df["Target"].value_counts()
percentual = df["Target"].value_counts(normalize=True).mul(100).round(1)
print(pd.DataFrame({"alunos": contagem, "%": percentual}))

ordem = ["Dropout", "Enrolled", "Graduate"]
fig, ax = plt.subplots(figsize=(5, 3.5))
ax.bar(ordem, [contagem[c] for c in ordem], color=["#c0392b", "#f39c12", "#27ae60"])
for i, c in enumerate(ordem):
    ax.text(i, contagem[c], f"{contagem[c]}\n({percentual[c]}%)", ha="center", va="bottom")
ax.set_ylabel("Nº de alunos")
ax.set_title("Distribuição do target")
ax.set_ylim(0, contagem.max() * 1.2)
fig.savefig(GRAFICOS / "eda_distribuicao_target.png")
plt.show()
# Leitura: se as classes forem desiguais, a acurácia sozinha engana (usar F1 macro e recall por classe).

## 3. Atributos categóricos (códigos sem ordem)

Estas colunas guardam **códigos** (curso, modo de candidatura etc.). Os números não têm ordem, então serão tratados com one-hot. A tabela mostra quantas categorias cada uma tem e quantas são raras (menos de 1% dos alunos), que serão agrupadas em "outros".

In [ ]:
CATEGORICAS = [
    "Marital status", "Application mode", "Course", "Daytime/evening attendance",
    "Previous qualification", "Nacionality", "Mother's qualification",
    "Father's qualification", "Mother's occupation", "Father's occupation",
]
ausentes = [c for c in CATEGORICAS if c not in df.columns]
assert not ausentes, f"Colunas não encontradas (conferir nomes): {ausentes}"

linhas = []
for c in CATEGORICAS:
    freq = df[c].value_counts(normalize=True)
    linhas.append({
        "coluna": c,
        "categorias": len(freq),
        "categoria_mais_comum_%": round(freq.iloc[0] * 100, 1),
        "categorias_raras(<1%)": int((freq < 0.01).sum()),
    })
pd.DataFrame(linhas)

## 4. Atributos numéricos

In [ ]:
NUMERICAS = [c for c in df.columns if c not in CATEGORICAS + ["Target"] and df[c].nunique() > 2]
BINARIAS = [c for c in df.columns if c not in CATEGORICAS + ["Target"] and df[c].nunique() <= 2]
print("Numéricas:", len(NUMERICAS), "| Binárias:", len(BINARIAS))
print("Binárias:", BINARIAS)
df[NUMERICAS].describe().T.round(2)

In [ ]:
# Escalas muito diferentes (notas, contagens, taxas) justificam a padronização antes da MLP
fig, axes = plt.subplots(4, 4, figsize=(14, 10))
for ax, c in zip(axes.ravel(), NUMERICAS[:16]):
    ax.hist(df[c], bins=25, color="#34495e")
    ax.set_title(c, fontsize=8)
    ax.tick_params(labelsize=7)
for ax in axes.ravel()[len(NUMERICAS[:16]):]:
    ax.axis("off")
fig.suptitle("Distribuição dos atributos numéricos (primeiros 16)")
fig.tight_layout()
fig.savefig(GRAFICOS / "eda_histogramas_numericas.png")
plt.show()

## 5. Relação dos atributos com o target

Desempenho acadêmico nos dois semestres tende a ser o sinal mais forte. Conferir nos dados.

In [ ]:
col_aprov = "Curricular units 2nd sem (approved)"
fig, ax = plt.subplots(figsize=(6, 4))
ax.boxplot([df.loc[df["Target"] == c, col_aprov] for c in ordem], tick_labels=ordem)
ax.set_ylabel(col_aprov)
ax.set_title("Disciplinas aprovadas no 2º semestre por classe")
fig.savefig(GRAFICOS / "eda_aprovadas_2sem_por_classe.png")
plt.show()

In [ ]:
# Correlação de cada atributo numérico/binário com o target codificado (0=Dropout, 1=Enrolled, 2=Graduate).
# Atenção: a correlação com um target de 3 classes é só um guia grosseiro, não uma prova de importância.
y_cod = df["Target"].map({c: i for i, c in enumerate(ordem)})
sem_cat = [c for c in df.columns if c not in CATEGORICAS + ["Target"]]
corr_target = df[sem_cat].corrwith(y_cod).sort_values()
print(corr_target.round(3).to_string())

fig, ax = plt.subplots(figsize=(7, 9))
corr_target.plot.barh(ax=ax, color="#2980b9")
ax.set_title("Correlação com o target (0/1/2)")
ax.tick_params(labelsize=7)
fig.savefig(GRAFICOS / "eda_correlacao_target.png")
plt.show()

In [ ]:
# Atributos altamente correlacionados entre si (redundância possível, ex.: 1º e 2º semestre)
corr = df[sem_cat].corr().abs()
pares = (corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
             .stack().sort_values(ascending=False))
print("Pares com |correlação| >= 0.8:")
print(pares[pares >= 0.8].round(3).to_string())

## 6. Atributos sensíveis e disponibilidade temporal

Gênero, nacionalidade, estado civil, necessidades educacionais especiais e idade são atributos sensíveis: podem produzir decisões discriminatórias se o modelo for usado na prática. Aqui só medimos como a evasão varia entre grupos (o debate vai para o relatório).

In [ ]:
def taxa_por_grupo(coluna):
    t = pd.crosstab(df[coluna], df["Target"], normalize="index").mul(100).round(1)
    t["n"] = df[coluna].value_counts()
    return t[ordem + ["n"]]

for c in ["Gender", "Educational special needs", "International", "Debtor", "Scholarship holder"]:
    print(f"\n{c}")
    print(taxa_por_grupo(c).to_string())

In [ ]:
# Idade na matrícula por classe
print(df.groupby("Target")["Age at enrollment"].describe().round(1).loc[ordem])

**Disponibilidade temporal (para o relatório):** os atributos `Curricular units 1st sem (...)` e `Curricular units 2nd sem (...)` só existem *depois* que o aluno começa o curso. Um modelo que os usa não prevê evasão "na matrícula", e sim ao longo do 1º/2º semestres. Isso precisa ser discutido, pois infla o desempenho em comparação com um cenário de previsão precoce.

## 7. Decisões sobre os atributos

Registradas em `docs/dataset.md` (tabela atributo a atributo). Resumo do que a EDA acima deve confirmar:

- códigos categóricos → one-hot, com categorias raras agrupadas em "outros";
- atributos numéricos → padronização (fit só no treino);
- todos os atributos são mantidos por padrão; qualquer descarte (ex.: `Nacionality`, se quase todos tiverem o mesmo valor) é decidido pelo grupo com base nas tabelas acima e registrado em `COLUNAS_DESCARTADAS` em `src/preprocessamento.py`.